# Parte 2 — Auditoria do Modelo A (Q2 + Q3)

## Contexto

A **Central 1746** usa o **modelo A** em produção para classificar automaticamente o texto dos chamados e encaminhá-los ao serviço correto. Antes de avaliar o **modelo B** (Parte 3), é necessário auditar o desempenho do modelo atual: quanto ele acerta, onde erra e com que grau de confiança.

Este notebook analisa as predições do modelo A sobre os **5.000 chamados rotulados** (`dados/chamados_com_predicoes.csv`), comparando `pred_modelo_a` com `categoria_real` (ground truth).

> **Nota:** os dados são sintéticos — simulam o comportamento estatístico de um sistema real, sem uso de dados de cidadãos.

## Objetivo

1. **Q2 — Desempenho com incerteza:** quantificar métricas globais e por categoria, com **intervalos de confiança (95%)**, justificando a escolha das métricas diante do **desbalanceamento** entre classes.
2. **Q3 — Onde o modelo falha?:** investigar modos de falha (matriz de confusão, subgrupos, calibração da confiança), formular hipóteses sobre causas e discutir o **impacto prático** no encaminhamento dos chamados.

## Dados utilizados

| Coluna | Descrição |
|--------|-----------|
| `categoria_real` | Rótulo humano (ground truth) |
| `pred_modelo_a` | Categoria prevista pelo modelo A |
| `conf_modelo_a` | Confiança declarada pelo modelo A (0 a 1) |
| `texto` | Input do classificador (usado para derivar `texto_len`) |
| `canal` | Subgrupo operacional (app, telefone, portal) |

*Colunas do modelo B (`pred_modelo_b`, `conf_modelo_b`) ficam reservadas para a Parte 3.*

## Conexão com a Parte 1 (EDA)

A análise exploratória levantou hipóteses que validamos aqui:

| Hipótese | Origem na EDA |
|----------|----------------|
| Classes desbalanceadas reduzem **recall** nas minoritárias (ex.: Sinalização)? | §§1–2 |
| Textos **curtos** (≤60 caracteres) têm pior desempenho que **longos** (≥150)? | §3 |
| Desempenho por **canal** difere pouco, salvo efeito de **volume** no app? | §4 |
| Erros seguem **pares semanticamente próximos** (matriz de confusão)? | a inferir na auditoria |

## Decisões metodológicas

| Decisão | Escolha |
|---------|---------|
| Métricas globais | Acurácia + macro-F1 |
| Por categoria | Recall, precision, F1 |
| Intervalos de confiança | Bootstrap estratificado (B=1000, 95%) |
| Matriz de confusão | Normalizada por linha (recall) |
| Subgrupos | Canal e faixa/quartil de `texto_len` |
| Figuras | `results/figures/auditoria/` |

## Roteiro da análise

1. **Setup e carregamento** — dados, colunas derivadas e validações  
2. **Q2 — Métricas globais** — acurácia e macro-F1 com IC  
3. **Q2 — Métricas por categoria** — recall, precision e F1 com IC  
4. **Q3 — Matriz de confusão** — heatmap e principais pares de erro  
5. **Q3 — Subgrupos** — desempenho por canal e comprimento do texto  
6. **Q3 — Calibração** — confiança do modelo A em acertos vs erros  
7. **Síntese** — modos de falha, impacto prático e validação das hipóteses da EDA


## Setup

Imports, caminhos e constantes usados em toda a auditoria. Figuras da Parte 2 vão em `results/figures/auditoria/`.

Fixamos `RANDOM_SEED` e `N_BOOTSTRAP` para reprodutibilidade do bootstrap (Q2).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# Exibe figuras no notebook e permite savefig no disco
%matplotlib inline

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "dados" / "chamados_com_predicoes.csv"
# Figuras da Parte 2 → results/figures/auditoria/
FIG_DIR = ROOT / "results" / "figures" / "auditoria"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
CI_LEVEL = 0.95

sns.set_theme(style="whitegrid")


## Carregamento e preparação para auditoria

Carregamos o mesmo corpus da Parte 1 e criamos colunas usadas nas seções Q2 e Q3:

- **`acerto`:** `pred_modelo_a == categoria_real`
- **`texto_len`:** comprimento do texto (caracteres)
- **`faixa_texto`:** faixas da EDA (≤60 / 61–149 / ≥150) — análise **primária** de subgrupo
- **`quartil_texto`:** quartis de `texto_len` — análise **complementar** (spec)

Validamos integridade do dataset antes de calcular métricas.


In [ ]:
df = pd.read_csv(DATA_PATH)

# Sanity checks na carga bruta
assert df.shape == (5000, 10)
assert df.isna().sum().sum() == 0
assert df["id_chamado"].is_unique
assert df["conf_modelo_a"].between(0, 1).all()

# Colunas derivadas
df["texto_len"] = df["texto"].str.len()
df["data_abertura"] = pd.to_datetime(df["data_abertura"])
df["acerto"] = df["pred_modelo_a"] == df["categoria_real"]
df["faixa_texto"] = pd.cut(
    df["texto_len"],
    bins=[0, 60, 149, np.inf],
    labels=["curto (≤60)", "médio (61–149)", "longo (≥150)"],
    include_lowest=True,
)
df["quartil_texto"] = pd.qcut(df["texto_len"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

assert (df["texto_len"] > 0).all()
assert set(df["categoria_real"]) == set(df["pred_modelo_a"])

LABELS = sorted(df["categoria_real"].unique())
y_true = df["categoria_real"]
y_pred = df["pred_modelo_a"]

n_erros = int((~df["acerto"]).sum())
acc_bruta = df["acerto"].mean()

resumo = pd.Series(
    {
        "registros": len(df),
        "acuracia_bruta_modelo_a": round(acc_bruta, 4),
        "acertos": int(df["acerto"].sum()),
        "erros": n_erros,
        "n_categorias": df["categoria_real"].nunique(),
        "conf_media": round(df["conf_modelo_a"].mean(), 3),
    }
)

display(resumo)
display(
    df[
        [
            "categoria_real",
            "pred_modelo_a",
            "conf_modelo_a",
            "acerto",
            "texto_len",
            "faixa_texto",
            "canal",
        ]
    ].head()
)


### Interpretação

- **5.000 chamados** válidos, sem nulos nem IDs duplicados — base consistente para métricas e bootstrap.
- O modelo A acerta **77,3%** dos casos nesta amostra (**1.136 erros**) — ponto de partida; o IC formal vem na seção Q2.
- **8 categorias** em `categoria_real` e `pred_modelo_a` — domínios alinhados, sem rótulos fora do vocabulário.
- Colunas `acerto`, `faixa_texto` e `quartil_texto` prontas para Q3 (subgrupos e hipóteses da EDA).

**Próximo passo:** Q2 — justificativa das métricas e bootstrap estratificado.


## Q2 — Justificativa das métricas e método do bootstrap

### Por que acurácia + macro-F1?

A EDA mostrou **desbalanceamento** entre as 8 categorias. A **acurácia** é interpretável para gestão (“quantos % dos chamados o modelo encaminha certo?”), mas pode **mascarar falhas em classes menores**.

O **macro-F1** é a média **não ponderada** do F1 de cada classe: trata todas as categorias igualmente, independentemente do suporte. Assim, mau desempenho em uma classe rara puxa a métrica para baixo — o que a acurácia sozinha pode esconder.

Por categoria reportamos **precision, recall e F1**: no encaminhamento, recall baixo significa que chamados reais daquela categoria estão sendo enviados para o serviço errado.

### Bootstrap estratificado (IC 95%)

1. Em cada reamostra `b = 1…B`, sorteamos com reposição **dentro de cada** `categoria_real`, preservando o tamanho de cada classe.
2. Calculamos as métricas na reamostra.
3. O IC 95% são os **percentis 2,5 e 97,5** da distribuição bootstrap.

Parâmetros: `B = 1000`, `seed = 42`, nível = 95%. A estratificação evita reamostras sem exemplos de classes raras e deixa o IC por categoria mais estável.


## Q2 — Função de bootstrap e métricas

Um único loop coleta métricas **globais** e **por categoria** (mesma seed), para eficiência e coerência dos ICs.


In [ ]:
def stratified_resample(df, strata_col, rng):
    """Reamostra com reposição preservando o tamanho de cada estrato."""
    parts = []
    for _, group in df.groupby(strata_col, sort=False):
        idx = rng.choice(group.index.to_numpy(), size=len(group), replace=True)
        parts.append(df.loc[idx])
    return pd.concat(parts, ignore_index=True)


def compute_metrics(y_true_s, y_pred_s, labels):
    acc = accuracy_score(y_true_s, y_pred_s)
    macro = f1_score(y_true_s, y_pred_s, average="macro", labels=labels, zero_division=0)
    p, r, f, s = precision_recall_fscore_support(
        y_true_s, y_pred_s, labels=labels, zero_division=0
    )
    return acc, macro, p, r, f, s


def bootstrap_all_metrics(df, labels, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    acc_point, macro_point, p0, r0, f0, s0 = compute_metrics(
        df["categoria_real"], df["pred_modelo_a"], labels
    )

    acc_boot = np.empty(n_boot)
    macro_boot = np.empty(n_boot)
    p_boot = np.empty((n_boot, len(labels)))
    r_boot = np.empty((n_boot, len(labels)))
    f_boot = np.empty((n_boot, len(labels)))

    for b in range(n_boot):
        sample = stratified_resample(df, "categoria_real", rng)
        acc_b, macro_b, p_b, r_b, f_b, _ = compute_metrics(
            sample["categoria_real"], sample["pred_modelo_a"], labels
        )
        acc_boot[b] = acc_b
        macro_boot[b] = macro_b
        p_boot[b] = p_b
        r_boot[b] = r_b
        f_boot[b] = f_b

    alpha = 1 - CI_LEVEL
    lo_q, hi_q = 100 * (alpha / 2), 100 * (1 - alpha / 2)

    global_tbl = pd.DataFrame(
        {
            "metrica": ["acuracia", "macro_f1"],
            "estimativa": [acc_point, macro_point],
            "ic95_low": [
                np.percentile(acc_boot, lo_q),
                np.percentile(macro_boot, lo_q),
            ],
            "ic95_high": [
                np.percentile(acc_boot, hi_q),
                np.percentile(macro_boot, hi_q),
            ],
        }
    )

    cat_tbl = pd.DataFrame(
        {
            "categoria": labels,
            "support": s0.astype(int),
            "precision": p0,
            "recall": r0,
            "f1": f0,
            "precision_ic95_low": np.percentile(p_boot, lo_q, axis=0),
            "precision_ic95_high": np.percentile(p_boot, hi_q, axis=0),
            "recall_ic95_low": np.percentile(r_boot, lo_q, axis=0),
            "recall_ic95_high": np.percentile(r_boot, hi_q, axis=0),
            "f1_ic95_low": np.percentile(f_boot, lo_q, axis=0),
            "f1_ic95_high": np.percentile(f_boot, hi_q, axis=0),
        }
    ).sort_values("f1")

    return global_tbl, cat_tbl, acc_boot, macro_boot


print(f"Rodando bootstrap estratificado B={N_BOOTSTRAP}, seed={RANDOM_SEED}…")
global_metrics, cat_metrics, acc_boot, macro_boot = bootstrap_all_metrics(df, LABELS)

# Sanity checks
acc_manual = (df["pred_modelo_a"] == df["categoria_real"]).mean()
assert abs(global_metrics.loc[0, "estimativa"] - acc_manual) < 1e-12
assert (
    global_metrics.loc[0, "ic95_low"]
    <= global_metrics.loc[0, "estimativa"]
    <= global_metrics.loc[0, "ic95_high"]
)
assert (
    global_metrics.loc[1, "ic95_low"]
    <= global_metrics.loc[1, "estimativa"]
    <= global_metrics.loc[1, "ic95_high"]
)
f1_mean_check = cat_metrics["f1"].mean()
assert abs(global_metrics.loc[1, "estimativa"] - f1_mean_check) < 1e-9
print("Sanity checks OK.")


### Métricas globais com IC 95%


In [ ]:
display(global_metrics.round(4))


### Métricas por categoria com IC 95%

Tabela ordenada por F1 (classes mais frágeis primeiro).


In [ ]:
display(cat_metrics.round(3))


### Interpretação (Q2)

- **Acurácia ≈ 0,773** e **macro-F1 ≈ 0,770**, com ICs 95% estreitos (~±0,01–0,02) — desempenho global estável nesta amostra.
- A classe mais frágil é **`esgoto_vazamento`** (recall ≈ **0,58**, F1 ≈ **0,64**): ~4 em 10 vazamentos reais são classificados como outra categoria.
- **`sinalizacao`** (menor suporte, n=227) tem F1 ≈ **0,77** — a hipótese EDA de “minoritária = recall baixo” **não se confirma** para sinalização; o problema principal não é só volume de classe.
- Classes fortes: **`iluminacao_publica`**, **`estacionamento_irregular`**, **`coleta_lixo`** (F1 ≳ 0,81).
- ICs de recall/F1 são um pouco mais largos em classes menores — interpretar `sinalizacao` com cautela.

**Próximo passo:** Q3 — matriz de confusão para ver *para onde* vão os erros de `esgoto_vazamento`.


## Q3 — Matriz de confusão e pares de erro

Heatmap **normalizado por linha**: cada linha soma 100% e mostra o **recall** da classe real e para quais categorias o modelo A desvia os erros.


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
cm_row = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_row,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=LABELS,
    yticklabels=LABELS,
    ax=ax,
    vmin=0,
    vmax=1,
)
ax.set_xlabel("Predição (modelo A)")
ax.set_ylabel("Categoria real")
ax.set_title("Matriz de confusão — modelo A (normalizada por linha)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_matriz_confusao.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Top pares de confusão (somente erros)
rows = []
for i, real in enumerate(LABELS):
    n_real = cm[i].sum()
    for j, pred in enumerate(LABELS):
        if i == j:
            continue
        n = int(cm[i, j])
        if n == 0:
            continue
        rows.append(
            {
                "real": real,
                "predito": pred,
                "n": n,
                "pct_da_classe_real": 100 * n / n_real,
            }
        )

top_conf = pd.DataFrame(rows).sort_values("n", ascending=False).head(10)
display(top_conf.round(1))

fig, ax = plt.subplots(figsize=(9, 5))
labels_bar = top_conf["real"] + " → " + top_conf["predito"]
ax.barh(labels_bar[::-1], top_conf["n"].values[::-1], color="steelblue")
ax.set_xlabel("Número de erros")
ax.set_title("Top 10 pares de confusão — modelo A")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_top_confusoes.png", dpi=200, bbox_inches="tight")
plt.show()


### Exemplos de texto nos principais pares de erro

Amostra de chamados reais nos 3 pares mais frequentes — para hipóteses sobre causa (proximidade semântica vs texto curto/ambíguo).


In [ ]:
def sample_pair(real, pred, n=2):
    mask = (df["categoria_real"] == real) & (df["pred_modelo_a"] == pred)
    cols = ["texto", "texto_len", "conf_modelo_a", "canal"]
    return df.loc[mask, cols].head(n)


pares_foco = [
    ("esgoto_vazamento", "buraco_via"),
    ("iluminacao_publica", "buraco_via"),
    ("iluminacao_publica", "esgoto_vazamento"),
]

for real, pred in pares_foco:
    n = int(
        ((df["categoria_real"] == real) & (df["pred_modelo_a"] == pred)).sum()
    )
    print(f"\n=== {real} → {pred} (n={n}) ===")
    display(sample_pair(real, pred, n=2))


### Interpretação (matriz de confusão)

- O modo de falha dominante é **`esgoto_vazamento` → `buraco_via`**: **221** erros (**~36%** dos esgotos reais). Categorias semanticamente próximas (via pública / infraestrutura no solo) — hipótese de **confusão lexical** (bueiro, rua, tampa).
- Outros pares relevantes envolvem **`iluminacao_publica`** desviando para `buraco_via` / `esgoto_vazamento` — parte dos exemplos são textos **muito curtos** (“Rua escura…”, “Lâmpada queimada…”), com poucas pistas.
- **Impacto prático:** vazamento de esgoto encaminhado como buraco atrasa equipe de saneamento e mantém risco sanitário; volume absoluto alto (221) torna este o principal risco operacional do modelo A.

**Próximo passo:** subgrupos por comprimento do texto e canal.


## Q3 — Desempenho por subgrupos

### Comprimento do texto

- **Primário:** faixas da EDA (`faixa_texto`) — testa a hipótese de pior desempenho em textos curtos (≤60).
- **Complementar:** quartis de `texto_len` (pedido da spec).


In [ ]:
def subgroup_table(col):
    g = (
        df.groupby(col, observed=True)
        .agg(
            n=("acerto", "size"),
            acuracia=("acerto", "mean"),
            n_erros=("acerto", lambda x: int((~x).sum())),
        )
        .reset_index()
    )
    g["pct_erros_globais"] = 100 * g["n_erros"] / n_erros
    return g


tbl_faixa = subgroup_table("faixa_texto")
tbl_quartil = subgroup_table("quartil_texto")

print("Faixas EDA (primário)")
display(tbl_faixa.round(3))
print("Quartis de texto_len (complementar)")
display(tbl_quartil.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(tbl_faixa["faixa_texto"].astype(str), tbl_faixa["acuracia"], color="steelblue")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Acurácia")
axes[0].set_title("Acurácia por faixa de texto (EDA)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(tbl_quartil["quartil_texto"].astype(str), tbl_quartil["acuracia"], color="teal")
axes[1].set_ylim(0, 1)
axes[1].set_title("Acurácia por quartil de texto_len")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_acuracia_por_texto_len.png", dpi=200, bbox_inches="tight")
plt.show()


### Canal

A EDA mostrou mix de classes similar entre canais; aqui medimos **acurácia** e **erros absolutos** (impacto operacional do volume do app).


In [ ]:
tbl_canal = subgroup_table("canal").sort_values("n", ascending=False)
display(tbl_canal.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(tbl_canal["canal"], tbl_canal["acuracia"], color="steelblue")
ax.set_ylim(0, 1)
ax.set_ylabel("Acurácia")
ax.set_title("Acurácia do modelo A por canal")
for i, row in tbl_canal.reset_index(drop=True).iterrows():
    ax.text(i, row["acuracia"] + 0.02, f"n={int(row['n'])}\nerros={int(row['n_erros'])}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_acuracia_por_canal.png", dpi=200, bbox_inches="tight")
plt.show()


### Interpretação (subgrupos)

- **Textos curtos (≤60):** acurácia ≈ **0,59** vs **≈0,84** nos longos (≥150) — diferença de ~25 p.p. Hipótese da EDA **confirmada**. Quase **metade dos erros** (576/1136) está nos curtos, embora sejam só ~28% do corpus.
- Quartis reforçam o padrão: **Q1** (textos mais curtos) ≈ **0,59**; Q2–Q4 ≈ **0,82–0,85**.
- **Canal:** acurácias próximas (**0,76–0,78**). App concentra **568 erros** (maior volume absoluto), não por pior taxa relativa. Hipótese EDA de “canal muda pouco” **confirmada**; priorizar mitigação por **texto curto**, não por canal.

**Próximo passo:** calibração da confiança declarada pelo modelo A.


## Q3 — Calibração da confiança (`conf_modelo_a`)

O modelo A declara uma confiança em [0, 1]. Se estiver bem calibrada, erros deveriam concentrar-se em baixas confianças — útil para **fila de revisão humana**.


In [ ]:
conf_by_acerto = df.groupby("acerto")["conf_modelo_a"].agg(["mean", "median", "std", "count"])
conf_by_acerto.index = conf_by_acerto.index.map({True: "acerto", False: "erro"})
display(conf_by_acerto.round(3))

df["conf_bin"] = pd.cut(
    df["conf_modelo_a"],
    bins=np.arange(0.6, 1.01, 0.1),
    include_lowest=True,
)
calib = (
    df.groupby("conf_bin", observed=True)
    .agg(
        n=("acerto", "size"),
        taxa_erro=("acerto", lambda x: (~x).mean()),
        conf_media=("conf_modelo_a", "mean"),
    )
    .reset_index()
)
display(calib.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(calib["conf_bin"].astype(str), calib["taxa_erro"], color="indianred")
ax.set_ylabel("Taxa de erro")
ax.set_xlabel("Faixa de conf_modelo_a")
ax.set_title("Taxa de erro por faixa de confiança — modelo A")
ax.set_ylim(0, max(0.4, calib["taxa_erro"].max() + 0.05))
fig.tight_layout()
fig.savefig(FIG_DIR / "05_confianca_vs_erro.png", dpi=200, bbox_inches="tight")
plt.show()


### Interpretação (calibração)

- Confiança média em **acertos ≈ 0,923** e em **erros ≈ 0,923** — praticamente **idêntica**.
- A taxa de erro **não cai** conforme a confiança sobe (faixas 0,8–0,9 e 0,9–1,0 têm erro ~**23%**). O score **não serve** como limiar confiável para triagem humana neste corpus.
- **Impacto prático:** não dá para “revisar só baixa confiança” — a mitigação deve mirar **subgrupos de falha** (textos curtos; confusão esgoto→buraco), não o valor de `conf_modelo_a`.

**Próximo passo:** síntese das hipóteses da EDA e modos de falha.


## Síntese — hipóteses, modos de falha e impacto

### Validação das hipóteses da Parte 1

| Hipótese EDA | Status | Evidência | Impacto prático |
|--------------|--------|-----------|-----------------|
| Classes desbalanceadas → recall baixo nas minoritárias (ex. Sinalização) | **Parcialmente refutada** | `sinalizacao` (n=227) tem F1≈0,77; o pior recall é `esgoto_vazamento` (0,58), classe de suporte médio | Não priorizar só “classes raras”; olhar confusões específicas |
| Textos curtos (≤60) pior que longos (≥150) | **Confirmada** | Acc ≈0,59 vs ≈0,84; 576 dos 1136 erros nos curtos | Formulário mínimo / pedir mais detalhe no app reduz risco |
| Canal muda pouco (só volume do app) | **Confirmada** | Acc 0,76–0,78; app tem mais erros absolutos por volume | Mitigar por conteúdo do texto, não por canal |
| Erros em pares semanticamente próximos | **Confirmada** | `esgoto_vazamento`→`buraco_via` = 221 (~36% da classe) | Encaminhamento errado atrasa saneamento; baseline para Parte 3 |

### Principais modos de falha (quantificados)

1. **Confusão esgoto → buraco** — 221 casos; maior risco sanitário/operacional.
2. **Textos curtos** — ~25 p.p. a menos de acurácia vs longos; metade dos erros.
3. **Confiança não calibrada** — `conf_modelo_a` não discrimina acerto/erro; inviabiliza triagem só por score.

### Limitações

- Dados **sintéticos** — ICs e padrões valem neste corpus; validar em produção real.
- ICs por categoria são mais largos em classes pequenas (`sinalizacao`).
- Esta auditoria **não** implica que o modelo B seja melhor — isso é a Parte 3.

### Próximo passo

Notebook `03_comparacao_e_recomendacao.ipynb`: comparar A vs B (teste pareado) usando estes modos de falha como baseline — em especial se B reduz a confusão esgoto→buraco e o gap em textos curtos.
